# Chronos-2 P0b: finite-candidate oracle screen

Evaluates the frozen constant, single-known-future, leave-one-out, and historical-correlation top-k policies on **calibration origins only**. P0a artifacts are integrity-checked as prerequisites, while all policies are rerun together on the current GPU for matched numerical comparison. It never instantiates sealed evaluation origins. Results are `screening_only`.

Use an **A100** if available. P0b is several times larger than P0a. Each completed `(task, origin)` is written atomically to Google Drive, so reconnecting and running all cells resumes safely.

In [ ]:
import importlib
import os
import subprocess
import sys
from pathlib import Path

REPO = Path('/content/covariate-safe-tsfm')
FEV_COMMIT = '38007871dcf6dc6b04aed3a54d9cd86678d48d0b'
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        os.environ['HF_TOKEN'] = hf_token
except Exception:
    pass

if not REPO.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/FlyMe2star/covariate-safe-tsfm.git', str(REPO)],
        check=True,
    )
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)],
    check=True,
)
subprocess.run(
    [
        sys.executable, '-m', 'pip', 'install', '-q',
        'chronos-forecasting==2.2.2',
        f'git+https://github.com/autogluon/fev.git@{FEV_COMMIT}',
    ],
    check=True,
)
SOURCE_ROOT = str(REPO / 'src')
if SOURCE_ROOT not in sys.path:
    sys.path.insert(0, SOURCE_ROOT)
importlib.invalidate_caches()
covsafe = importlib.import_module('covsafe')
print('Repository ready:', REPO)
print('Git commit:', subprocess.check_output(
    ['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True
).strip())
print('HF token available:', bool(os.environ.get('HF_TOKEN')))

In [ ]:
import torch

assert torch.cuda.is_available(), (
    'Select Runtime > Change runtime type > GPU, then restart.'
)
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda)
print('Torch:', torch.__version__)

In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
PRIVATE_ROOT = Path(
    '/content/drive/MyDrive/covariate-safe-tsfm/private_manifests'
)
P0A_ROOT = PRIVATE_ROOT / 'p0a'
P0B_ROOT = PRIVATE_ROOT / 'p0b'
assert P0A_ROOT.exists(), 'P0a artifacts are missing from Google Drive.'
P0B_ROOT.mkdir(parents=True, exist_ok=True)
print('P0a input root:', P0A_ROOT)
print('P0b durable root:', P0B_ROOT)

In [ ]:
import json

from covsafe.chronos2_p0b import run_chronos2_p0b
from covsafe.p0b import EXPECTED_P0B_CONFIG_HASH

print('Frozen P0b config hash:', EXPECTED_P0B_CONFIG_HASH)
report = run_chronos2_p0b(REPO, P0A_ROOT, P0B_ROOT)
print(json.dumps(report, indent=2, ensure_ascii=False, default=str))

## Return artifact

Send the final JSON containing `h2_screen`. If disconnected, reconnect, select a GPU, and run all cells; completed origins print `RESUME`. Do not run the TimesFM notebook until this report has been reviewed.